In [ ]:
import pandas as pd
import numpy as np
import os, json
import fitz
from pathlib import Path
import time
from collections import Counter
from utils import Helper
utils = Helper()

TEXT OR SCANNED

In [ ]:
folder_pdf = r"C:\Users\kaustubh.keny\Documents\Quarterly Results 2026 Q1"
#2279
def page_text_or_scanned(page):

    text = page.get_text("text").strip()
    page_rect = page.rect
    page_area = page_rect.width * page_rect.height

    if page_area <= 0:
        return "scanned"

    image_area = 0
    for img in page.get_images(full=True):
        try:
            xref = img[0]
            for rect in page.get_image_rects(xref):
                clipped = rect & page_rect
                if clipped.is_empty:
                    continue

                rect_area = clipped.width * clipped.height
                # Ignore small logos
                if rect_area > page_area * 0.05:
                    image_area += rect_area

        except Exception:
            continue

    image_coverage = min(image_area / page_area, 1.0)
    blocks = page.get_text("blocks")
    text_blocks = [
        block for block in blocks if len(block) >= 5 and str(block[4]).strip()
    ]

    num_text_blocks = len(text_blocks)

    # Strong text page
    if len(text) > 100 and num_text_blocks >= 3 and image_coverage < 0.8:
        return "text"
    # Strong scanned page
    if image_coverage > 0.8 and len(text) < 100:
        return "scanned"
    # OCR scanned page
    if image_coverage > 0.9 and num_text_blocks <= 2:
        return "scanned"
    return "text" if len(text) > 100 else "scanned"



all_data = []
pdf_registry =  os.listdir(folder_pdf)
total_pdfs = len(pdf_registry)
print(f"TOTAL PDF(S): {total_pdfs}")

for idx, file in enumerate(pdf_registry[2280:]):
    file_path = os.path.join(folder_pdf, file)
    stem = Path(file_path).name
    print(f"{idx + 2270}/{total_pdfs}: {stem}")

    doc = fitz.open(file_path)
    for idx, page in enumerate(doc):
        res = page_text_or_scanned(page)
        all_data.append({
            "pdf_name":stem,
            "page_n":idx +1,
            "type":res
        })
        
    doc.close()

In [4]:
fpath = Path(folder_pdf)
df = pd.DataFrame(all_data)
df.to_excel(f"{fpath.stem}_TYPE2.xlsx" ,index=False)

In [ ]:
#CUT PDF + MERGE
import fitz
from pathlib import Path

input_folder = r"C:\Users\kaustubh.keny\Projects\INPUTS\ANNUAL_REPORTS\ANNUAL_REPORTS_2026"
output_pdf = "FIRST2.pdf"

merged_doc = fitz.open()

for pdf_file in sorted(Path(input_folder).glob("*.pdf")):
    try:
        src = fitz.open(pdf_file)
        #if src.page_count > 0:
        merged_doc.insert_pdf(src, from_page=0, to_page=0)

        src.close()
        print(f"Added {pdf_file.name}")

    except Exception as e:
        print(f"Error processing {pdf_file.name}: {e}")

print(f"Total pages: {merged_doc.page_count}")

merged_doc.save(output_pdf)
merged_doc.close()

print(f"Saved: {output_pdf}")

In [ ]:
import re
import fitz
from pathlib import Path

input_folder = r"C:\Users\kaustubh.keny\Projects\INPUTS\ANNUAL_REPORTS\ANNUAL_REPORTS_2026"
output_pdf = "INDEX261.pdf"
# "Corporate Information","CORPORATE OVERVIEW","STATUTORY REPORTS","FINANCIAL STATEMENTS","Contents","Corporate Governance","Business Responsibility","Sustainability Report","Standalone","Consolidated"
terms = [
    r"CORPORATE\s+OVERVIEW",
    r"STATUTORY\s+REPORTS?",
    r"FINANCIAL\s+(?:STATEMENTS?|REPORTS?)",
    r"\bCONTENTS\b",
    r"CORPORATE\s+GOVERNANCE\s+REPORT",
    r"BUSINESS\s+RESPONSIBILITY",
    r"SUSTAINABILITY\s+REPORT",
    r"STANDALONE(?:\s+FINANCIAL)?",
    r"CONSOLIDATED(?:\s+FINANCIAL)?",
]

pattern = re.compile("|".join(terms), re.I)
master_doc = fitz.open()

for pdf_file in sorted(Path(input_folder).glob("*.pdf")):

    try:
        doc = fitz.open(pdf_file)

        search_limit = min(50, doc.page_count) # Content section is before in the PDF
        matched_page = None
        for page_num in range(search_limit):

            text = doc[page_num].get_text("text")
            matches = len(pattern.findall(text))
            if matches >= 3:
                matched_page = page_num
                break

        if matched_page is not None:

            start_page = master_doc.page_count
            master_doc.insert_pdf(
                doc,
                from_page=matched_page,
                to_page=matched_page
            )

            page = master_doc[start_page]
            page.insert_text(
                (40, 30),
                f"SOURCE: {pdf_file.stem}",
                fontsize=12,
                color=(1, 0, 0)
            )
        else:
            print(f"No match: {pdf_file.name}")

        doc.close()

    except Exception as e:
        print(f"Error: {pdf_file.name} -> {e}")

